In [2]:
import warp as wp
import numpy as np

wp.init()

Warp 1.11.0.dev20251115 initialized:
   Git commit: 62778e81d556f6573509dc314a49794370940d55
   CUDA Toolkit 12.8, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA GeForce RTX 5090" (31 GiB, sm_120, mempool enabled)
   Kernel cache:
     /home/fenglong/.cache/warp/1.11.0.dev20251115


In [ ]:
TILE_SIZE = wp.constant(64)
TILE_THREADS = 32

@wp.kernel
def compute(a: wp.array2d(dtype=float), b: wp.array2d(dtype=float)):

	# obtain our block index
	i = wp.tid()

	# load a row from global memory
	t = wp.tile_load(a[i], TILE_SIZE)

	# cooperatively compute the sum of the tile elements; s is a single element tile
	s = wp.tile_sum(t)

	# store s in global memory
	wp.tile_store(b[i], s)

N = 10

a_np = np.arange(N).reshape(-1, 1) * np.ones((1, 256), dtype=float)
a = wp.array(a_np, dtype=float)
b = wp.zeros((N,1), dtype=float)

wp.launch_tiled(compute, dim=[a.shape[0]], inputs=[a, b], block_dim=TILE_THREADS)

print(f"b = {b[:,0]}")

Module __main__ 684fb78 load on device 'cuda:0' took 233.18 ms  (compiled)
b = [  0.  64. 128. 192. 256. 320. 384. 448. 512. 576.]


In [ ]:

def random_spd(n, eps=1e-2):
	A = np.random.randn(n, n).astype(np.float64)
	return (A @ A.T + eps * np.eye(n).astype(np.float64))

TILE_SIZE = 32           # Single tile size

A_np = random_spd(TILE_SIZE)

A = wp.array(A_np, dtype=wp.float64, device="cuda")
L = wp.zeros((TILE_SIZE, TILE_SIZE), dtype=wp.float64, device="cuda")


@wp.kernel
def cholesky_kernel(A: wp.array2d(dtype=wp.float64),
					L: wp.array2d(dtype=wp.float64)):
	# Load entire matrix into a tile
	a = wp.tile_load(A, shape=(TILE_SIZE, TILE_SIZE))
	
	# Perform Cholesky decomposition on the tile
	l = wp.tile_cholesky(a)
	
	# Store result back to global memory
	wp.tile_store(L, l)


# Launch with dim=[1, 1] since we're processing the whole matrix as one tile
wp.launch_tiled(
	kernel=cholesky_kernel,
	dim=[1, 1],
	inputs=[A, L],
	block_dim=128,
	device="cuda"
)

L_np = np.linalg.cholesky(A_np.astype(np.float64))

print("Warp L (top-left corner):")
print(L.numpy()[:5, :5])

print("\nNumPy L (top-left corner):")
print(L_np[:5, :5])

# Error
print("\nFrobenius Error:", np.linalg.norm(L.numpy() - L_np.astype(np.float64)))

NameError: name 'np' is not defined

### GEMM
Examples taken from: https://developer.nvidia.com/blog/introducing-tile-based-programming-in-warp-1-5-0/

In [21]:
# tile size
TILE_M = wp.constant(32)
TILE_N = wp.constant(32)
TILE_K = wp.constant(32)

# num threads per-tile
TILE_THREADS = 64

@wp.kernel
def tile_gemm(A: wp.array2d(dtype=float), B: wp.array2d(dtype=float), C: wp.array2d(dtype=float)):

    # output tile index
    i, j = wp.tid()

    sum = wp.tile_zeros(shape=(TILE_M, TILE_N), dtype=wp.float32)

    K = A.shape[1]
    count = int(K / TILE_K)
    for k in range(0, count):
        a = wp.tile_load(A, shape=(TILE_M, TILE_K), offset=(i*TILE_M, k*TILE_K))
        b = wp.tile_load(B, shape=(TILE_K, TILE_N), offset=(k*TILE_K, j*TILE_N))
        # sum += a*b
        wp.tile_matmul(a, b, sum, alpha=1.)

    wp.tile_store(C, sum, offset=(i*TILE_M, j*TILE_N))
    

# generate some tile aligned matrix dimensions
M = 1024
K = 1024
N = 1024

rng = np.random.default_rng(42)
A = rng.random((M, K), dtype=np.float32)
B = rng.random((K, N), dtype=np.float32)
C = np.zeros((M, N), dtype=np.float32)

A_wp = wp.array(A)
B_wp = wp.array(B)
C_wp = wp.array(C)

with wp.Tape() as tape:
	wp.launch_tiled(
		tile_gemm,
		dim=(int(M / TILE_M), int(N / TILE_N)),
		inputs=[A_wp, B_wp, C_wp],
		block_dim=TILE_THREADS)

assert(np.allclose(C_wp.numpy(), A@B))

print("Example matrix multiplication passed")

Module __main__ a0d8cad load on device 'cuda:0' took 3011.75 ms  (compiled)
Example matrix multiplication passed


In [17]:
%%timeit
wp.launch_tiled(
	tile_gemm,
	dim=(int(M / TILE_M), int(N / TILE_N)),
	inputs=[A_wp, B_wp, C_wp],
	block_dim=TILE_THREADS)
wp.synchronize()

104 μs ± 143 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


Batched GEMM

In [40]:
@wp.kernel
def tile_batch_gemm(A: wp.array3d(dtype=float), B: wp.array3d(dtype=float), C: wp.array3d(dtype=float)):
	b, i, j = wp.tid()  # the first dimension is the batch
	sum = wp.tile_zeros(shape=(1, TILE_M, TILE_N), dtype=wp.float32)

	count = int(A.shape[2] / TILE_K)
	for k in range(0, count):
		a_tile = wp.tile_load(A, shape=(1, TILE_M, TILE_K), offset=(b, i*TILE_M, k*TILE_K))
		b_tile = wp.tile_load(B, shape=(1, TILE_K, TILE_N), offset=(b, k*TILE_K, j*TILE_N))
		# sum += a*b
		wp.tile_matmul(a_tile[0], b_tile[0], sum[0], alpha=1.)

	wp.tile_store(C, sum, offset=(b, i*TILE_M, j*TILE_N))

# The improved version
@wp.kernel
def tile_batch_gemm2(A: wp.array3d(dtype=float), B: wp.array3d(dtype=float), C: wp.array3d(dtype=float)):
	b, i, j = wp.tid()  # the first dimension is the batch
	sum = wp.tile_zeros(shape=(TILE_M, TILE_N), dtype=wp.float32)

	count = int(A.shape[2] / TILE_K)
	for k in range(0, count):
		# Load 2D slices directly from the batch dimension
		a_tile = wp.tile_load(A[b], shape=(TILE_M, TILE_K), offset=(i*TILE_M, k*TILE_K))
		b_tile = wp.tile_load(B[b], shape=(TILE_K, TILE_N), offset=(k*TILE_K, j*TILE_N))
		# sum += a*b
		wp.tile_matmul(a_tile, b_tile, sum, alpha=1.)

	wp.tile_store(C[b], sum, offset=(i*TILE_M, j*TILE_N))

# Most optimized - flatten batch dimension into output grid
@wp.kernel
def tile_batch_gemm3(A: wp.array3d(dtype=float), B: wp.array3d(dtype=float), C: wp.array3d(dtype=float),
					 n_tiles_m: int, n_tiles_n: int):
	idx = wp.tid()
	
	# Compute batch index and tile indices from flattened index
	tiles_per_batch = n_tiles_m * n_tiles_n
	b = idx // tiles_per_batch
	tile_idx = idx % tiles_per_batch
	i = tile_idx // n_tiles_n
	j = tile_idx % n_tiles_n
	
	sum = wp.tile_zeros(shape=(TILE_M, TILE_N), dtype=wp.float32)

	count = int(A.shape[2] / TILE_K)
	for k in range(0, count):
		a_tile = wp.tile_load(A[b], shape=(TILE_M, TILE_K), offset=(i*TILE_M, k*TILE_K))
		b_tile = wp.tile_load(B[b], shape=(TILE_K, TILE_N), offset=(k*TILE_K, j*TILE_N))
		wp.tile_matmul(a_tile, b_tile, sum, alpha=1.)

	wp.tile_store(C[b], sum, offset=(i*TILE_M, j*TILE_N))

# generate some tile aligned matrix dimensions
B = 128
M = 256
K = 128
N = 512

rng = np.random.default_rng(42)
A_batch = rng.random((B, M, K), dtype=np.float32)
B_batch = rng.random((B, K, N), dtype=np.float32)
C_batch = np.zeros((B, M, N), dtype=np.float32)

A_wp = wp.array(A_batch)
B_wp = wp.array(B_batch)
C_wp = wp.array(C_batch)
wp.launch_tiled(
	tile_batch_gemm,
	dim=(B, int(M / TILE_M), int(N / TILE_N)),
	inputs=[A_wp, B_wp, C_wp],
	block_dim=TILE_THREADS)

for b in range(B):
	assert(np.allclose(C_wp.numpy()[b], A_batch[b]@B_batch[b]))

wp.launch_tiled(
	tile_batch_gemm2,
	dim=(B, int(M / TILE_M), int(N / TILE_N)),
	inputs=[A_wp, B_wp, C_wp],
	block_dim=TILE_THREADS)

for b in range(B):
	assert(np.allclose(C_wp.numpy()[b], A_batch[b]@B_batch[b]))

# Test version 3
n_tiles_m = int(M / TILE_M)
n_tiles_n = int(N / TILE_N)
total_tiles = B * n_tiles_m * n_tiles_n
C_wp = wp.array(C_batch)
wp.launch_tiled(
	tile_batch_gemm3,
	dim=total_tiles,
	inputs=[A_wp, B_wp, C_wp, n_tiles_m, n_tiles_n],
	block_dim=TILE_THREADS)

for b in range(B):
	assert(np.allclose(C_wp.numpy()[b], A_batch[b]@B_batch[b]))

print("All batched matrix multiplication versions passed")

Module __main__ 6cc9b91 load on device 'cuda:0' took 7405.80 ms  (compiled)
All batched matrix multiplication versions passed
All batched matrix multiplication versions passed


In [41]:
%%timeit
wp.launch_tiled(
	tile_batch_gemm,
	dim=(B, int(M / TILE_M), int(N / TILE_N)),
	inputs=[A_wp, B_wp, C_wp],
	block_dim=TILE_THREADS)
wp.synchronize()

652 μs ± 2.16 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [42]:
%%timeit
wp.launch_tiled(
	tile_batch_gemm2,
	dim=(B, int(M / TILE_M), int(N / TILE_N)),
	inputs=[A_wp, B_wp, C_wp],
	block_dim=TILE_THREADS)
wp.synchronize()

217 μs ± 1.64 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [43]:
%%timeit 
wp.launch_tiled(
	tile_batch_gemm3, 
	dim=total_tiles, 
	inputs=[A_wp, B_wp, C_wp, n_tiles_m, n_tiles_n], 
	block_dim=TILE_THREADS) 
wp.synchronize()

236 μs ± 2.34 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [ ]:
import warp as wp
import numpy as np

wp.init()

# -----------------------------------------------------
# 1. Generate a random SPD matrix
# -----------------------------------------------------

def random_spd(n, eps=1e-2):
	A = np.random.randn(n, n).astype(np.float64)
	return (A @ A.T + eps * np.eye(n).astype(np.float64))

N = 32                  # matrix dimension
T = 16                  # tile size (must divide N)

A_np = random_spd(N)


# -----------------------------------------------------
# 2. Allocate Warp arrays
# -----------------------------------------------------
A = wp.array(A_np, dtype=wp.float64, device="cuda")
L = wp.zeros((N, N), dtype=wp.float64, device="cuda")


# -----------------------------------------------------
# 3. Kernel to perform tile-based Cholesky
# -----------------------------------------------------
@wp.kernel
def cholesky_tiled(A: wp.array2d(dtype=wp.float64),
				   L: wp.array2d(dtype=wp.float64)):
	ti, tj = wp.tid()     # tile row, tile col

	# Create a local tile
	tile = wp.tile_zeros((T, T), dtype=wp.float64)

	# Load tile
	wp.tile_load(tile, A, (ti*T, tj*T))

	# Only process diagonal tiles (simple version)
	if ti == tj:
		wp.tile_cholesky(tile)

	# Store back result
	wp.tile_store(L, tile, (ti*T, tj*T))


# -----------------------------------------------------
# 4. Launch tiled kernel
# -----------------------------------------------------
grid = (N // T, N // T)

wp.launch(
	kernel=cholesky_tiled,
	dim=grid,
	inputs=[A, L],
	device="cuda"
)

# -----------------------------------------------------
# 5. Compare with NumPy result
# -----------------------------------------------------
L_np = np.linalg.cholesky(A_np)

print("Warp L (top-left corner):")
print(L.numpy()[:5, :5])

print("\nNumPy L (top-left corner):")
print(L_np[:5, :5])

# Error
print("\nFrobenius Error:", np.linalg.norm(L.numpy() - L_np))

Warp 1.11.0.dev20251115 initialized:
   Git commit: 62778e81d556f6573509dc314a49794370940d55
   CUDA Toolkit 12.8, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA GeForce RTX 5090" (31 GiB, sm_120, mempool enabled)
   Kernel cache:
     /home/fenglong/.cache/warp/1.11.0.dev20251115
Module __main__ f419c47 load on device 'cuda:0' took 0.45 ms  (error)


WarpCodegenError: Error while parsing function "cholesky_tiled" at /tmp/ipykernel_81938/3982436874.py:39:
    wp.tile_load(tile, A, (ti*T, tj*T))
;Couldn't find function overload for 'tile_load' that matched inputs with types: [tile(shape=(16, 16), dtype=float64), array(ndim=2, dtype=float64), tuple(int32, int32)]